In [2]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Set

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from mylib.maze_graph import CP_DSPs, maze1_graph

Graph = Dict[int, List[int]]

# -------------------------
# 12x12 geometry (row-major)
# -------------------------
NROWS, NCOLS = 12, 12
# --- reward shaping ---
GOAL_REWARD = 1.0
DEAD_END_PENALTY = -0.2   # <- tune this (e.g. -0.05 to -0.5)


def bin_to_rc(b: int) -> Tuple[int, int]:
    """bin id (1..144) -> (row, col)"""
    b0 = b - 1
    return b0 // NCOLS, b0 % NCOLS

def rc_to_bin(r: int, c: int) -> Optional[int]:
    """(row, col) -> bin id (1..144)"""
    if 0 <= r < NROWS and 0 <= c < NCOLS:
        return r * NCOLS + c + 1
    return None

def vec_from_to(b_from: int, b_to: int) -> Tuple[int, int]:
    """grid displacement from b_from -> b_to"""
    r0, c0 = bin_to_rc(b_from)
    r1, c1 = bin_to_rc(b_to)
    return (r1 - r0, c1 - c0)

def rot_left(dr: int, dc: int) -> Tuple[int, int]:
    """rotate a heading vector left"""
    return (-dc, dr)

def rot_right(dr: int, dc: int) -> Tuple[int, int]:
    """rotate a heading vector right"""
    return (dc, -dr)

def heading_from(prev_b: int, curr_b: int) -> Tuple[int, int]:
    """heading vector inferred from prev_b -> curr_b"""
    dr, dc = vec_from_to(prev_b, curr_b)
    if abs(dr) + abs(dc) != 1:
        return (0, 0)
    return (dr, dc)

def edge_allowed(g: Graph, curr_b: int, next_b: int, blocked_directed: Set[Tuple[int,int]]) -> bool:
    """whether curr_b -> next_b is allowed (graph edge exists + not blocked)"""
    return (curr_b, next_b) not in blocked_directed and (next_b in g[curr_b])

def forward_neighbors(g: Graph, curr_b: int, prev_b: Optional[int], blocked_directed: Set[Tuple[int,int]]) -> List[int]:
    """Allowed neighbors excluding backtracking to prev_b."""
    options = [nb for nb in g[curr_b] if edge_allowed(g, curr_b, nb, blocked_directed)]
    if prev_b is None:
        return options
    return [nb for nb in options if nb != prev_b]

def is_dead_end(g: Graph, curr_b: int, prev_b: Optional[int], blocked_directed: Set[Tuple[int,int]]) -> bool:
    """Dead end = no forward option (only can go back)."""
    if prev_b is None:
        return False
    return len(forward_neighbors(g, curr_b, prev_b, blocked_directed)) == 0

# Egocentric wall outcomes: (L,S,R) each 1=wall,0=open
def egocentric_walls(
    g: Graph,
    curr_b: int,
    heading: Tuple[int,int],
    blocked_directed: Set[Tuple[int,int]]
) -> Tuple[int,int,int]:
    """
    Egocentric wall outcomes at curr_b given heading:
      returns (left_is_wall, straight_is_wall, right_is_wall)
    """
    dr, dc = heading
    if abs(dr) + abs(dc) != 1:
        return (1, 1, 1)  # unknown heading -> conservative

    r, c = bin_to_rc(curr_b)
    dirs = [rot_left(dr, dc), (dr, dc), rot_right(dr, dc)]
    out = []
    for drr, dcc in dirs:
        next_b = rc_to_bin(r + drr, c + dcc)
        if next_b is None:
            out.append(1)
        else:
            out.append(0 if edge_allowed(g, curr_b, next_b, blocked_directed) else 1)
    return (out[0], out[1], out[2])

def hamming3(a: Tuple[int,int,int], b: Tuple[int,int,int]) -> int:
    """Hamming distance between two (L,S,R) triplets"""
    return int(a[0]!=b[0]) + int(a[1]!=b[1]) + int(a[2]!=b[2])

# -------------------------
# SR + SARSA agent
# -------------------------
@dataclass
class TDSR_SARSA_Agent:
    n_states: int
    gamma: float = 0.97
    alpha_sr: float = 0.10
    alpha_q: float = 0.10
    tau: float = 0.20          # softmax temperature
    beta_novel: float = 0.8    # novelty bonus scale
    seed: int = 0

    def __post_init__(self) -> None:
        self.rng = np.random.default_rng(self.seed)
        self.M = np.zeros((self.n_states, self.n_states), dtype=np.float32)   # SR: M[s, s']
        self.I = np.eye(self.n_states, dtype=np.float32)                      # one-hot basis
        self.Q = np.zeros((self.n_states, self.n_states), dtype=np.float32)   # Q[s, s_to] (action = go to s_to)
        self.Nsa = np.zeros((self.n_states, self.n_states), dtype=np.int32)   # visit counts N[s, s_to]

    def td_sr_update(self, s_idx: int, s_next_idx: int, terminal: bool) -> None:
        """TD(0) SR update on transition s -> s_next"""
        target = self.I[s_idx].copy()
        if not terminal:
            target += self.gamma * self.M[s_next_idx]
        self.M[s_idx] += self.alpha_sr * (target - self.M[s_idx])

    def sarsa_update(
        self,
        s_idx: int,
        a_to_idx: int,
        reward: float,
        s_next_idx: int,
        a_next_to_idx: Optional[int],
        terminal: bool
    ) -> None:
        """SARSA update for Q(s, a_to) where a_to means 'go to state a_to_idx'"""
        q_sa = self.Q[s_idx, a_to_idx]
        if terminal or a_next_to_idx is None:
            target = reward
        else:
            target = reward + self.gamma * self.Q[s_next_idx, a_next_to_idx]
        self.Q[s_idx, a_to_idx] = q_sa + self.alpha_q * (target - q_sa)

    def novelty_bonus(self, s_idx: int, a_to_idx: int) -> float:
        """novelty = 1/sqrt(N+1)"""
        return 1.0 / np.sqrt(float(self.Nsa[s_idx, a_to_idx]) + 1.0)

    def choose_action(
        self,
        g: Graph,
        curr_b: int,
        prev_b: Optional[int],
        blocked_directed: Set[Tuple[int,int]],
        allow_backtrack: bool = True,
    ) -> Optional[int]:
        """
        Choose next bin next_b among allowed neighbors using softmax over:
            score(next_b) = Q[curr_b -> next_b] + beta * novelty(curr_b -> next_b)

        prev_b is used only to optionally suppress backtracking.
        """
        s_idx = curr_b - 1

        candidate_next_bs = [nb for nb in g[curr_b] if edge_allowed(g, curr_b, nb, blocked_directed)]
        if not candidate_next_bs:
            return None

        if (not allow_backtrack) and prev_b is not None and len(candidate_next_bs) > 1:
            no_back = [nb for nb in candidate_next_bs if nb != prev_b]
            if no_back:
                candidate_next_bs = no_back

        scores = []
        for next_b in candidate_next_bs:
            a_to_idx = next_b - 1
            score = float(self.Q[s_idx, a_to_idx]) + self.beta_novel * float(self.novelty_bonus(s_idx, a_to_idx))
            scores.append(score)

        scores = np.array(scores, dtype=np.float64)
        logits = scores / max(self.tau, 1e-8)
        logits -= logits.max()
        probs = np.exp(logits)
        probs /= probs.sum()

        next_b = int(self.rng.choice(candidate_next_bs, p=probs))
        return next_b

# -------------------------
# Path (R1) extraction in a tree (unique)
# -------------------------
from collections import deque

def bfs_path(g: Graph, start_b: int, goal_b: int) -> List[int]:
    """unique path if g is a tree"""
    if start_b == goal_b:
        return [start_b]
    parent = {start_b: None}
    q = deque([start_b])
    while q:
        curr_b = q.popleft()
        for next_b in g[curr_b]:
            if next_b not in parent:
                parent[next_b] = curr_b
                if next_b == goal_b:
                    q.clear()
                    break
                q.append(next_b)
    if goal_b not in parent:
        raise ValueError(f"No path from {start_b} to {goal_b}")
    path = []
    curr_b = goal_b
    while curr_b is not None:
        path.append(curr_b)
        curr_b = parent[curr_b]
    path.reverse()
    return path

# -------------------------
# Training on episodes start_b -> goal_b (same policy class)
# -------------------------
def train_agent(
    g: Graph,
    n_states: int = 144,
    start_b: int = 1,
    goal_b: int = 144,
    n_episodes: int = 8000,
    max_steps: int = 600,
    blocked_directed: Optional[Set[Tuple[int,int]]] = None,
    seed: int = 0,
) -> Tuple[TDSR_SARSA_Agent, List[int]]:
    if blocked_directed is None:
        blocked_directed = set()

    agent = TDSR_SARSA_Agent(n_states=n_states, seed=seed)

    # unique track (reference)
    R1 = bfs_path(g, 1, 144)

    for ep in tqdm(range(n_episodes)):
        curr_b = start_b
        prev_b: Optional[int] = None

        next_b = agent.choose_action(g, curr_b, prev_b, blocked_directed, allow_backtrack=True)
        if next_b is None:
            continue

        steps_taken = 0
        terminal = False

        while not terminal:
            steps_taken += 1

            s_idx = curr_b - 1
            a_to_idx = next_b - 1  # action is "go to next_b"

            # count visit for novelty
            agent.Nsa[s_idx, a_to_idx] += 1

            # sparse reward: only when entering goal_b
            if next_b == goal_b:
                reward = GOAL_REWARD
            else:
                # If we just arrived at a dead end (only option is to go back), penalize
                reward = DEAD_END_PENALTY if is_dead_end(g, next_b, curr_b, blocked_directed) else 0.0
                
            terminal = (next_b == goal_b)

            # SR update uses transition curr_b -> next_b
            agent.td_sr_update(s_idx, next_b - 1, terminal=terminal)

            if terminal:
                agent.sarsa_update(s_idx, a_to_idx, reward, next_b - 1, a_next_to_idx=None, terminal=True)
                break

            # advance environment
            next_curr_b = next_b
            next_prev_b = curr_b

            # pick next action from next state (SARSA)
            next_next_b = agent.choose_action(g, next_curr_b, next_prev_b, blocked_directed, allow_backtrack=True)
            if next_next_b is None:
                # dead end -> terminal (0 extra reward)
                agent.sarsa_update(s_idx, a_to_idx, reward, next_curr_b - 1, a_next_to_idx=None, terminal=True)
                break

            agent.sarsa_update(
                s_idx, a_to_idx, reward,
                next_curr_b - 1, a_next_to_idx=(next_next_b - 1),
                terminal=False
            )

            # shift pointers
            prev_b, curr_b, next_b = next_prev_b, next_curr_b, next_next_b

    return agent, R1

# -------------------------
# Retrieval/localization restricted to R1 using egocentric mismatch
# -------------------------
def localize_on_R1_egocentric(
    g: Graph,
    agent: TDSR_SARSA_Agent,
    R1: List[int],
    start_true_b: int = 99,
    goal_true_b: int = 144,
    blocked_directed: Optional[Set[Tuple[int,int]]] = None,
    beta_match: float = 4.0,
    max_steps: int = 400,
) -> Tuple[List[int], List[int], np.ndarray]:
    """
    True movement uses the SAME learned policy (Q + novelty),
    but inference is constrained to R1 and based on egocentric (L,S,R) walls.
    """
    if blocked_directed is None:
        blocked_directed = set()

    R1 = list(R1)
    K = len(R1)
    idxR = {b:i for i,b in enumerate(R1)}

    # belief over candidate bins on R1
    bel = np.ones(K, dtype=np.float64) / K

    true_traj: List[int] = []
    inferred: List[int] = []
    beliefs: List[np.ndarray] = []

    # true dynamics
    curr_b = start_true_b
    prev_b: Optional[int] = None

    # candidate "previous bin" for heading on R1: assume candidate is moving along R1 toward goal
    cand_prev_b: List[Optional[int]] = []
    for b in R1:
        i = idxR[b]
        cand_prev_b.append(R1[i-1] if i > 0 else None)

    while 1:
        true_traj.append(curr_b)

        # true heading (prev_b -> curr_b), or sample one step to define initial heading
        if prev_b is not None:
            head_true = heading_from(prev_b, curr_b)
        else:
            sampled_next_b = agent.choose_action(g, curr_b, prev_b, blocked_directed, allow_backtrack=True)
            head_true = (0,0) if sampled_next_b is None else heading_from(curr_b, sampled_next_b)

        obs_true = egocentric_walls(g, curr_b, head_true, blocked_directed)

        # compare with each candidate R1 bin's predicted egocentric walls
        mismatch = np.zeros(K, dtype=np.float64)
        for k, cand_b in enumerate(R1):
            cand_prev = cand_prev_b[k]

            if cand_prev is not None:
                head_cand = heading_from(cand_prev, cand_b)
            else:
                sampled_next_b = agent.choose_action(g, cand_b, None, blocked_directed, allow_backtrack=True)
                head_cand = (0,0) if sampled_next_b is None else heading_from(cand_b, sampled_next_b)

            obs_pred = egocentric_walls(g, cand_b, head_cand, blocked_directed)
            mismatch[k] = hamming3(obs_true, obs_pred)

        # belief update
        bel *= np.exp(-beta_match * mismatch)
        z = bel.sum()
        bel = (np.ones(K) / K) if z < 1e-300 else (bel / z)

        map_idx = int(np.argmax(bel))
        inferred.append(R1[map_idx])
        beliefs.append(bel.copy())

        if curr_b == goal_true_b:
            break

        # true movement: same learned policy
        next_b = agent.choose_action(g, curr_b, prev_b, blocked_directed, allow_backtrack=True)
        if next_b is None:
            raise ValueError(f"True trajectory got stuck at {curr_b} with no allowed moves.")
        prev_b, curr_b = curr_b, next_b

    return true_traj, inferred, np.vstack(beliefs)

# -------------------------
# Custom-track plotting (unchanged except variable names)
# -------------------------
main_track = CP_DSPs[1][0]

def plot_real_vs_inferred_on_custom_track(
    main_track: list[int],
    true_traj: list[int],
    inferred: list[int],
    beliefs: np.ndarray | None = None,
    y_start_at_1: bool = True,
):
    idx = {b: i for i, b in enumerate(main_track)}

    T = min(len(true_traj), len(inferred))
    t = np.arange(T)

    real_y = np.array([idx.get(true_traj[i], np.nan) for i in range(T)], dtype=float)
    inf_y  = np.array([idx.get(inferred[i],  np.nan) for i in range(T)], dtype=float)

    if y_start_at_1:
        real_y = real_y + 1
        inf_y  = inf_y + 1

    plt.figure(figsize=(11, 6))
    plt.plot(t, real_y, linewidth=3, label="Real position (main-track index)")
    plt.plot(t, inf_y, "--", linewidth=2, label="Inferred place code (main-track index)")
    plt.xlabel("Time step")
    plt.ylabel("Index along provided main track")
    plt.title("Real vs inferred position (visualized on custom main-track order)")
    plt.grid(True)
    plt.legend()
    plt.show()

    valid = ~np.isnan(real_y) & ~np.isnan(inf_y)
    err = np.full(T, np.nan, dtype=float)
    err[valid] = np.abs(real_y[valid] - inf_y[valid])

    plt.figure(figsize=(11, 4.5))
    plt.plot(t, err, linewidth=2)
    plt.xlabel("Time step")
    plt.ylabel("Absolute error (index distance)")
    plt.title("Localization error on the custom main-track index")
    plt.grid(True)
    plt.show()

    if beliefs is not None:
        conf = beliefs[:T].max(axis=1)
        plt.figure(figsize=(11, 4.5))
        plt.plot(t, conf, linewidth=2)
        plt.xlabel("Time step")
        plt.ylabel("Max belief")
        plt.title("Retrieval confidence")
        plt.ylim(0, 1.05)
        plt.grid(True)
        plt.show()

    print("\nStep | Real bin | Real idx | Inferred bin | Inf idx")
    print("-"*58)
    for i in range(min(30, T)):
        ry = real_y[i]
        iy = inf_y[i]
        print(f"{i:4d} | {true_traj[i]:8d} | {ry:8.1f} | {inferred[i]:12d} | {iy:7.1f}")

# -------------------------
# Run
# -------------------------
if __name__ == "__main__":
    # Temporary walls only during retrieval
    blocked_directed_retrieval = {(23, 100), (100, 99)}  # keep as you wrote

    agent, R1 = train_agent(
        maze1_graph,
        start_b=1,
        goal_b=144,
        blocked_directed=set(),   # IMPORTANT: no extra walls during training
        n_episodes=1000,
        max_steps=10000,
        seed=0,
    )

100%|██████████| 1000/1000 [03:12<00:00,  5.20it/s]


In [ ]:
if 1:
    print("R1 length:", len(R1), "| 99 on R1:", (99 in set(R1)))

    true_traj, inferred, beliefs = localize_on_R1_egocentric(
        maze1_graph,
        agent,
        R1,
        start_true_b=23,
        goal_true_b=144,
        blocked_directed=blocked_directed_retrieval,  # wall applied only now
        beta_match=4.0,
        max_steps=10000,
    )

    print("True traj length:", len(true_traj), "| reached 144:", (true_traj[-1] == 144))
    plot_real_vs_inferred_on_custom_track(
        main_track=main_track,
        true_traj=true_traj,
        inferred=inferred,
        beliefs=beliefs,
        y_start_at_1=True,
    )

R1 length: 111 | 99 on R1: True
